In [ ]:
!pip install google-generativeai
!pip install orjson
!pip install prompt-toolkit


In [ ]:
import google.generativeai as genai
import re
import json

# -----------------------------
# Embed your Gemini API Key here
# -----------------------------
API_KEY = "XXXXXXXXXXXXXXXXX"
genai.configure(api_key=API_KEY)

# STEM Ladder settings
current_level = 1
score = 0
use_ai_answers = True

grade_level = "7th Grade"   # adjustable
subject = "Algebra"         # adjustable
max_difficulty = 10         # adjustable

model = genai.GenerativeModel("gemini-2.5-flash")

print("Welcome to the STEM Learning Ladder! Type 'exit' to quit.\n")

# Generate question from Gemini
def generate_question(level, grade, subject, max_diff):
    difficulty = min(level, max_diff)
    prompt = (
        f"Generate a {subject} math question for {grade} students at difficulty {difficulty} "
        f"(1=easy, {max_diff}=hard). "
        f"Return the result strictly in JSON with keys 'question' and 'answer'."
    )
    response = model.generate_content(prompt)
    raw_output = response.text

    # Try parsing JSON
    try:
        json_str = raw_output[raw_output.index("{"): raw_output.rindex("}")+1]
        question_data = json.loads(json_str)
        return question_data.get("question", "Error: no question"), question_data.get("answer", "Error: no answer")
    except Exception:
        return f"Parsing failed: {raw_output}", "N/A"

# --- Helper to extract numbers ---
def extract_number(text):
    nums = re.findall(r'-?\d+', str(text))  # handles negatives
    return int(nums[0]) if nums else None

# --- Main loop ---
while True:
    print(f"Difficulty Level: {current_level}/{max_difficulty} | Grade: {grade_level} | Subject: {subject}")

    # Generate fresh question
    question_text, correct_answer = generate_question(current_level, grade_level, subject, max_difficulty)
    print(f"Question: {question_text}")

    user_input = input("Your Answer: ")
    if user_input.lower() == "exit":
        print(f"Final score: {score}")
        break

    # AI-generated answer
    if use_ai_answers:
        answer_prompt = f"Solve this {subject} question: {question_text}. Provide only the numeric or symbolic answer."
        response = model.generate_content(answer_prompt)
        ai_answer = response.text
        print(f"AI Suggested Answer: {ai_answer}")

    # Check user's answer
    user_val = extract_number(user_input)
    correct_val = extract_number(correct_answer)

    if user_val is not None and correct_val is not None and user_val == correct_val:
        print("Correct")
        score += 1
        if current_level < max_difficulty:
            current_level += 1
    else:
        print(f"Incorrect. Correct answer: {correct_answer}")
        if current_level > 1:
            current_level -= 1

    print(f"Current Score: {score}\n")


Welcome to the STEM Learning Ladder! Type 'exit' to quit.

Difficulty Level: 1/10 | Grade: 7th Grade | Subject: Algebra
Question: What is the value of x in the equation: x + 8 = 15?
Your Answer: 7
AI Suggested Answer: 7
Correct
Current Score: 1

Difficulty Level: 2/10 | Grade: 7th Grade | Subject: Algebra
Question: If you subtract 7 from a number, the result is 15. What is the number?
Your Answer: 22
AI Suggested Answer: 22
Correct
Current Score: 2

Difficulty Level: 3/10 | Grade: 7th Grade | Subject: Algebra
Question: Maria bought some bags of apples. Each bag had 6 apples. She also bought 3 extra loose apples. If she has a total of 27 apples, how many bags of apples did she buy?


KeyboardInterrupt: Interrupted by user